# Basic figures for 311 service requests vs EMS heat calls

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:

import os
import shii
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import datetime as dt

APP_TOKEN = os.getenv('NYC_OPEN_DATA_APP_TOKEN')

## 1. Prep data

In [ ]:

all_311_df = shii.prepare_all_311_requests(APP_TOKEN, output_path='./311_calls.gpkg')
heat_inc_df = shii.prepare_ems_calls(APP_TOKEN, output_path='./ems_calls.csv')

In [ ]:
full_df = shii.preprocess_merge_df(heat_inc_df, all_311_df, summer_only=False)

In [ ]:
# Normalize variables by population
for col in ['ac','hydrant','power','ventilation','fhe', 'heat_ems_count']:
    full_df[col+'_norm'] = 100000*full_df[col]/full_df['population']

# Get total 311 counts

In [ ]:
cdta_grouped = full_df.groupby('cdta').mean()
cdta_grouped_sum = full_df.groupby('cdta').sum()
# Replace sums with mean values in cases where means don't make sense
per_cdta_vars = ['HVI_RANK', 'SURFACE_TEMP', 'MEDIAN_INCOME',
                 'GREENSPACE', 'PCT_HOUSEHOLDS_AC','PCT_BLACK_POP',
                 'population',]
cdta_grouped_sum[per_cdta_vars] = cdta_grouped[per_cdta_vars]


### Time series

In [ ]:
# Fig 2: Time series comparison
fig, axs = plt.subplots(2, 3, figsize=(12, 8))
ax_flat = axs.flatten()
start_date = '2023-01-01'
end_date = '2024-12-31'
# Heat counts
date_grouped['heat_ems_count_norm'].plot(
    xlim=(start_date, end_date),
    ax=axs[0, 0],
    xlabel='',
    ylabel='EMS Calls per 100k',
    ylim=(0, 1),
    title='Heat Illness EMS Calls'
    )

# Temp
date_grouped['tmax'].plot(
    xlim=(start_date, end_date),
    ax=axs[1, 0],
    color='darkred',
    ylabel='Temperature (C)',
    xlabel='',
    title='Max Temp'
    )

# 311 calls
ax_list = [axs[0, 1], axs[0,2], axs[1,1], axs[1,2]]
color_list = ['#D62828', '#FCBF49', '#0077B6', '#2A9D8F']
title_list = ['311 Hydrant', '311 Ventilation', '311 Power', '311 AC']

for i, col in enumerate(['hydrant_norm','ventilation_norm','ac_norm','power_norm']):
    date_grouped[col].plot(
        xlim=(start_date, end_date),
        ax=ax_list[i],
        color = color_list[i],
        xlabel='',
        ylabel='Requests per 100k people',
        title=title_list[i]
        )
plt.tight_layout()

### Scatter plots

In [ ]:
all_311_df.loc[all_311_df['request_type']=='ventilation'].groupby('agency_name').size()

In [ ]:
# Fig 3: Per CDTA comparisons with heat ems counts
fig, axs = plt.subplots(2,3, figsize=(12, 6))
color_list = ['#D62828', '#FCBF49', '#0077B6', '#2A9D8F']
cdta_grouped_sum.plot.scatter(
    x='MEDIAN_INCOME',
    y='heat_ems_count_norm',
    ax=axs[0, 0],
    color='green',
    title='Median Income vs. Heat Illness',
    ylabel='EMS Dispatches',
    xlabel='Median Income ($/year)',

    )
cdta_grouped_sum.plot.scatter(
    x='GREENSPACE',
    y='heat_ems_count_norm',
    ax=axs[1, 0],
    color='darkgreen',
    title='Greenspace vs. Heat Illness',
    ylabel='EMS Dispatches',
    xlabel='Greenspace (%)',
    )
cdta_grouped_sum.plot.scatter(
    x='hydrant_norm',
    y='heat_ems_count_norm',
    ax=axs[0, 1],
    color=color_list[0],
    title='311 Hydrant vs. Heat Illness',
    ylabel='EMS Dispatches (per 100k)',
    xlabel='311 Hydrant (per 100k)',
    )
cdta_grouped_sum.plot.scatter(
    x='ventilation_norm',
    y='heat_ems_count_norm',
    ax=axs[0, 2],
    color=color_list[1],
    title='311 Ventilation vs. Heat Illness',
    ylabel='EMS Dispatches (per 100k)',
    xlabel='311 Ventilation (per 100k)'
    )
cdta_grouped_sum.plot.scatter(
    x='fhe_norm',
    y='heat_ems_count_norm',
    ax=axs[1, 1],
    color='darkorange',
    title='311 FHE vs. Heat Illness',
    ylabel='EMS Dispatches (per 100k)',
    xlabel='311 FHE (per 100k)'
    )
cdta_grouped_sum.plot.scatter(
    x='ac_norm',
    y='heat_ems_count_norm',
    ax=axs[1, 2],
    color=color_list[3],
    title='311 AC vs. Heat Illness',
    ylabel='EMS Dispatches (per 100k)',
    xlabel='311 AC (per 100k)'
    )
fig.tight_layout()

### Maps

In [ ]:
cdta_map = shii.download_community_districts().rename(columns={'boro_cd':'cdta'}).set_index('cdta')[['geometry']]

In [ ]:
# Fig 1: Heat EMS times series and map
fig, axs = plt.subplots(1, 2, figsize=(15, 4))
date_grouped['heat_ems_count'].plot(
    ax=axs[0],
    xlabel='',
    ylabel='EMS Calls per day',
    title='Heat Illness EMS Calls per day',
    color='darkred'
    )


cdta_map.join(cdta_grouped_sum).plot(
    column='heat_ems_count',
    legend=True,
    ax=axs[1],
)
axs[1].set_title('Total Heat Illness EMS Calls, 2010-2024')
axs[1].set_xlabel('Lon')
axs[1].set_xlabel('Lat')
fig.show()